# Análisis Exploratorio de Movilidad Telefónica

Este notebook caracteriza la movilidad intermunicipal de la provincia de Barcelona y evalúa métodos para estimar el volumen de viajes que se esperan en una fecha determinada.

Fuente: fichero procesado M2.parquet, correspondiente a relaciones entre municipios distintos. La variable viajes representa el volumen agregado publicado para cada combinación de atributos; no identifica el modo de transporte, los recorridos individuales ni el horario del viaje. Las relaciones seleccionadas por corredor ferroviario son criterios territoriales, no recuentos de pasajeros de Rodalies.

Período: 01/01/2023–31/08/2025. En M2 hay 971 fechas con observaciones; faltan por completo 11/02/2024, 13/04/2025 y 20/05/2025. Los recuentos positivos menores de 15 viajes se publican como cero, lo que afecta sobre todo a flujos pequeños.


In [ ]:

from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holidays
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "data/processed/telefonica/M2.parquet").is_file())
M2_PATH = PROJECT_ROOT / "data/processed/telefonica/M2.parquet"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.rmove_general import cargar_catalogo
from analysis.rmove_indicators import (
    cargar_relaciones, evaluar_mediana, metricas_referencia,
    calcular_indicadores, MESES_EXCLUIDOS,
)
print("M2 encontrado:", M2_PATH.is_file())


## 1. Datos y alcance del análisis

Se consulta el fichero M2 consolidado mediante DuckDB, sin cargar sus más de 64 millones de filas íntegramente en memoria. El primer resumen permite verificar el período, la cobertura de fechas y el volumen acumulado de viajes.


In [ ]:
con = duckdb.connect()

# Obtener información general del dataset
resumen_m2 = con.execute(
    """
    SELECT
        COUNT(*) AS registros,
        MIN(day) AS primera_fecha,
        MAX(day) AS ultima_fecha,
        COUNT(DISTINCT day) AS dias_disponibles,
        SUM(viajes) AS total_viajes
    FROM read_parquet(?)
    """,
    [str(M2_PATH)]
).df()

display(resumen_m2)

## 2. Análisis temporal de la movilidad

El objetivo es entender la tendencia general y los patrones periódicos antes de evaluar desviaciones.

### 2.1 Evolución diaria por año

Se agregan los viajes de M2 para cada fecha y se representa la serie temporal de 2023, 2024 y 2025 por separado. 2025 solo incluye de enero a agosto, por lo que no es directamente comparable con años completos.

In [ ]:
# Calcular el volumen total de viajes por día
movilidad_diaria = con.execute(
    """
    SELECT
        day,
        SUM(viajes) AS viajes
    FROM read_parquet(?)
    GROUP BY day
    ORDER BY day
    """,
    [str(M2_PATH)]
).df()

display(movilidad_diaria.head())

print("Días analizados:", len(movilidad_diaria))


plt.figure(figsize=(14, 5))

plt.plot(
    movilidad_diaria["day"],
    movilidad_diaria["viajes"],
    linewidth=0.8
)

plt.title("Evolución diaria de la movilidad intermunicipal (2023–2025)")
plt.xlabel("Fecha")
plt.ylabel("Viajes diarios publicados")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:

# Añadir el año a partir de la fecha
movilidad_diaria["año"] = movilidad_diaria["day"].dt.year

# Crear tres gráficos, uno para cada año
fig, axes = plt.subplots(3, 1, figsize=(15, 11))

# Misma escala vertical para facilitar la comparación
limite_y = movilidad_diaria["viajes"].max() * 1.05

for ax, año in zip(axes, [2023, 2024, 2025]):

    datos_año = movilidad_diaria[
        movilidad_diaria["año"] == año
    ]

    ax.plot(
        datos_año["day"],
        datos_año["viajes"],
        linewidth=0.9
    )

    ax.set_title(f"Movilidad intermunicipal - {año}")
    ax.set_ylabel("Viajes diarios")
    ax.set_ylim(0, limite_y)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Fecha")

plt.tight_layout()
plt.show()


Las series muestran oscilaciones semanales recurrentes: la movilidad suele ser mayor en los dias laborables y menor los dias de fines de semana. También se aprecia reducción de viajes en verano (de forma estacional), especialmente en agosto.

Alguno mínimos puntuales destacan frente a lo que es habitual. Son fechas candidatas a investigar, pero no podemos atribuirlos directamente a festivos ni a incidencias ferroviarias sin cruzar las fechas con fuentes externas.

### 2.2 Estadística descriptiva de los viajes diarios

La función .describe() resume la distribución de los totales diarios de movilidad de M2. Cada una de sus 971 observaciones representa un dia, no una fila individual del fichero.

In [ ]:

estadisticas_diarias = movilidad_diaria["viajes"].describe()
display(estadisticas_diarias)


En las 971 fechas observadas, el promedio es de 3.796.704 viajes diarios y la mediana de 4.112.109. La media queda por debajo de la mediana, lo que sugiere la existencia de bastantes dias de movilidad relativamente baja (por ejemplo, fines de semana y vacaciones).

Los valores diarios van desde 1.465.533 hasta 5.133.535 viajes. La desviación estándar es de 786.746 viajes, lo que equivale a un coeficiente de variación aproximado de 20.67%. El 50% central de los dias se situa entre 3.127.009 y 4.451290 viajes.

Esto nos da a ver que la media global no es una referencia suficiente para detectar alteraciones, porque mezcla dias laborables, fines de semana, festivos y estaciones del año.

### 2.3 Distribución por día de la semana y año

Se comparan dos perspectivas: medias por dia de la semana y boxplots (dispersión y valores extremos). Una misma media puede ocultar dias muy diferentes, por lo que ambas són útiles para caracterizar la movilidad previa al modelado.

In [ ]:

# Asegurar que la fecha está en formato datetime
movilidad_diaria["day"] = pd.to_datetime(movilidad_diaria["day"])

# Crear las variables temporales
movilidad_diaria["año"] = movilidad_diaria["day"].dt.year
movilidad_diaria["dia_semana"] = movilidad_diaria["day"].dt.day_of_week

dias = [
    "Lunes", "Martes", "Miércoles",
    "Jueves", "Viernes", "Sábado", "Domingo"
]

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)

for ax, año in zip(axes, [2023, 2024, 2025]):

    datos_año = movilidad_diaria[
        movilidad_diaria["año"] == año
    ]

    medias = (
        datos_año
        .groupby("dia_semana")["viajes"]
        .mean()
        .reindex(range(7))
    )

    ax.bar(dias, medias / 1_000_000)

    ax.set_title(f"Movilidad semanal - {año}")
    ax.set_xlabel("Día de la semana")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Media de viajes diarios (millones)")

plt.suptitle("Media de viajes por día de la semana y año")
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)

for ax, año in zip(axes, [2023, 2024, 2025]):

    datos_año = movilidad_diaria[
        movilidad_diaria["año"] == año
    ]

    datos_boxplot = [
        datos_año.loc[
            datos_año["dia_semana"] == d, "viajes"
        ].dropna() / 1_000_000
        for d in range(7)
    ]

    ax.boxplot(
        datos_boxplot,
        tick_labels=dias,
        showmeans=True
    )

    ax.set_title(f"Distribución semanal - {año}")
    ax.set_xlabel("Día de la semana")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Viajes diarios (millones)")

plt.suptitle("Distribución de viajes por día de la semana y año")
plt.tight_layout()
plt.show()


Los gráficos por día de la semana muestran que la mediana y la dispersión cambian segun el calendario. Los valores atípicos inferiores en dias laborables pueden corresponder a festivos, vacaciones u otras circunstancias, aunque no se ha verificado.

En los boxplots, la línea interior de la caja es la mediana, el triangulo verde representa la media, la caja contiene el 50% central y los circulos son valores fuera de los bigotes. Un valor atípico no equivale a una incidencia ferroviaria.

In [ ]:
estadisticas_semanales = (
    movilidad_diaria
    .groupby("dia_semana")["viajes"]
    .agg(
        observaciones="count",
        media="mean",
        mediana="median",
        desviacion_estandar="std"
    )
    .reindex(range(7))
)

# Calcular el coeficiente de variación
estadisticas_semanales["cv_pct"] = (
    estadisticas_semanales["desviacion_estandar"]
    / estadisticas_semanales["media"]
    * 100
)

# Añadir los nombres de los días
dias = [
    "Lunes", "Martes", "Miércoles",
    "Jueves", "Viernes", "Sábado", "Domingo"
]

estadisticas_semanales["dia"] = dias

# Mostrar resultados
display(
    estadisticas_semanales[
        [
            "dia",
            "observaciones",
            "media",
            "mediana",
            "desviacion_estandar",
            "cv_pct"
        ]
    ].round(2)
)


In [ ]:

# Obtener las medias
media_jueves = estadisticas_semanales.loc[3, "media"]
media_domingo = estadisticas_semanales.loc[6, "media"]

# Diferencia absoluta
diferencia_viajes = media_jueves - media_domingo

# Diferencia porcentual respecto al jueves
diferencia_pct = diferencia_viajes / media_jueves * 100

print(f"Media de viajes los jueves: {media_jueves:,.0f}")
print(f"Media de viajes los domingos: {media_domingo:,.0f}")

print(f"\nDiferencia absoluta: {diferencia_viajes:,.0f} viajes")

print(
    f"Reducción porcentual del domingo respecto al jueves: "
    f"{diferencia_pct:.2f}%"
)


Los jueves presentan una media aproximada de 4,26 millones de viajes, mientras que los domingos registran 2,61 millones. Esto supone una reducción del 38,80 % respecto al volumen medio de los jueves.

La distancia entre ambos promedios confirma que es imprescindible comparar cada fecha con referencias compatibles con su dia de la semana.

In [ ]:

cv_lunes = estadisticas_semanales.loc[0, "cv_pct"]
cv_domingo = estadisticas_semanales.loc[6, "cv_pct"]

print(f"Coeficiente de variación del lunes: {cv_lunes:.2f}%")
print(f"Coeficiente de variación del domingo: {cv_domingo:.2f}%")

print(
    f"Diferencia de variabilidad relativa: "
    f"{cv_lunes - cv_domingo:.2f} puntos porcentuales"
)


El coeficiente de variación (CV) nos indica cuánto varian los datos en relación con su propia media:

$$
CV = \frac{\sigma}{\mu} \times 100
$$

Vemos que los lunes tambien presentan una mayor variabilidad relativa. Es decirm los domingos suelen tener un comportamiento mas estable respecto a su propia media, mientras que entre distintos lunes encontramos diferencias proporcionalmente mayores. Su movilidad es menos uniforme.

Esto es importante porque una desviación determinada puede ser relativamente normal para un dia de la semana, pero poco habitual para otro.


In [ ]:

plt.figure(figsize=(10, 5))

plt.bar(
    estadisticas_semanales["dia"],
    estadisticas_semanales["cv_pct"],
    color="steelblue"
)

plt.title("Variabilidad relativa de la movilidad por día de la semana")
plt.xlabel("Día de la semana")
plt.ylabel("Coeficiente de variación (%)")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


### 2.3 Síntesis del patrón semanal

La movilidad entre municipios de M2 presenta un patrón semanal marcado: predominan los volúmenes altos durante los días laborables y una reducción en fin de semana, especialmente los domingos. Los días laborables también registran ocasionalmente mínimos muy pronunciados.

Esto justifica utilizar referencias condicionadas por el día de la semana y estudiar después mes, vacaciones y festivos.

### 2.4 Estacionalidad menusal

El promedio diario de cada mes permite distinguir la estacionalidad de la variación semanal. Se observa una caída importante durante **agosto**, repetida en los años con ese mes disponible. El promedio mensual es una descripción, no una corrección automática de festivos ni un contraste causal.

In [ ]:
# Extraer el mes de cada observación
movilidad_diaria["mes"] = movilidad_diaria["day"].dt.month

# Calcular el promedio diario de viajes por mes y año
movilidad_mensual = (
    movilidad_diaria
    .groupby(["año", "mes"])["viajes"]
    .mean()
    .reset_index()
)

# Representar un gráfico para cada año
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)

for ax, año in zip(axes, [2023, 2024, 2025]):

    datos_año = movilidad_mensual[
        movilidad_mensual["año"] == año
    ]

    ax.plot(
        datos_año["mes"],
        datos_año["viajes"] / 1_000_000,
        marker="o"
    )

    ax.set_title(f"Movilidad mensual - {año}")
    ax.set_xlabel("Mes")
    ax.set_xticks(range(1, 13))
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Media diaria de viajes (millones)")

plt.suptitle("Evolución de la movilidad media diaria por mes y año")
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(11, 5))

for año in [2023, 2024, 2025]:

    datos_año = movilidad_mensual[
        movilidad_mensual["año"] == año
    ]

    plt.plot(
        datos_año["mes"],
        datos_año["viajes"] / 1_000_000,
        marker="o",
        linewidth=2,
        label=str(año)
    )

plt.title("Comparación de la movilidad media diaria por mes (2023–2025)")
plt.xlabel("Mes")
plt.ylabel("Media diaria de viajes (millones)")
plt.xticks(
    range(1, 13),
    ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
     "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
)
plt.legend(title="Año")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Análisis espacial de la movilidad entre municipios

Se identifican las principales relaciones origen-destino y los municipios con mayores volúmenes de salida y llegada, para conocer la estructura territorial.

### 3.1 Relaciones origen-destino con mayor volumen

Las relaciones se ordenan por viajes agregados en el período y se presentan también como media diaria. Como el total incluye solo los 971 días observados, la media diaria se calcula dividiendo por ese número de días, no por el número de filas de la relación.

In [ ]:
principales_relaciones = con.execute(
    """
    SELECT
        municipio_origen,
        municipio_origen_name,
        municipio_destino,
        municipio_destino_name,
        SUM(viajes) AS viajes_totales,
        SUM(viajes) / 971.0 AS media_diaria
    FROM read_parquet(?)
    GROUP BY
        municipio_origen,
        municipio_origen_name,
        municipio_destino,
        municipio_destino_name
    ORDER BY viajes_totales DESC
    LIMIT 20
    """,
    [str(M2_PATH)]
).df()

display(principales_relaciones)


In [ ]:

# Crear una etiqueta para cada relación
principales_relaciones["relacion"] = (
    principales_relaciones["municipio_origen_name"]
    + " → "
    + principales_relaciones["municipio_destino_name"]
)

# Ordenar para el gráfico
datos_plot = principales_relaciones.sort_values("media_diaria")

plt.figure(figsize=(12, 8))

plt.barh(
    datos_plot["relacion"],
    datos_plot["media_diaria"],
    color="steelblue"
)

plt.title("20 principales relaciones intermunicipales (2023–2025)")
plt.xlabel("Media diaria de viajes publicados")
plt.ylabel("Relación origen-destino")
plt.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()


### 3.2 Municipios con más viajes de salida y llegada

Barcelona destaca en ambos sentidos, seguida por municipios metropolitanos como L'Hospitalet de Llobregat, El Prat de Llobregat y Badalona. Esto sugiere que debemos observar corredores concretos y no solamente el total.

Los volumenes no demuestra que cada persona realice un viaje de ida y vuelta.

In [ ]:

# Municipios con mayor volumen de viajes como origen
principales_origenes = con.execute(
    """
    SELECT
        municipio_origen,
        municipio_origen_name,
        SUM(viajes) AS viajes_totales,
        SUM(viajes) / 971.0 AS media_diaria
    FROM read_parquet(?)
    GROUP BY municipio_origen, municipio_origen_name
    ORDER BY viajes_totales DESC
    LIMIT 15
    """,
    [str(M2_PATH)]
).df()

# Municipios con mayor volumen de viajes como destino
principales_destinos = con.execute(
    """
    SELECT
        municipio_destino,
        municipio_destino_name,
        SUM(viajes) AS viajes_totales,
        SUM(viajes) / 971.0 AS media_diaria
    FROM read_parquet(?)
    GROUP BY municipio_destino, municipio_destino_name
    ORDER BY viajes_totales DESC
    LIMIT 15
    """,
    [str(M2_PATH)]
).df()

display(principales_origenes)
display(principales_destinos)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Principales municipios de origen
origenes_plot = principales_origenes.sort_values("media_diaria")

axes[0].barh(
    origenes_plot["municipio_origen_name"],
    origenes_plot["media_diaria"],
    color="steelblue"
)

axes[0].set_title("Principales municipios de origen")
axes[0].set_xlabel("Media diaria de viajes")
axes[0].grid(axis="x", alpha=0.3)

# Principales municipios de destino
destinos_plot = principales_destinos.sort_values("media_diaria")

axes[1].barh(
    destinos_plot["municipio_destino_name"],
    destinos_plot["media_diaria"],
    color="darkorange"
)

axes[1].set_title("Principales municipios de destino")
axes[1].set_xlabel("Media diaria de viajes")
axes[1].grid(axis="x", alpha=0.3)

plt.suptitle("Principales municipios según origen y destino (2023–2025)")
plt.tight_layout()
plt.show()


## 4. Evolución temporal de una relación específica: Mataró --> Barcelona

Se selecciona esta relación como primer caso de estudio, por su interés territorial con la R1. 

Primero comprobamos la disponibilidad de fechas, después estimamos la movilidad habitual y cuantificamos la diferencia entre los valores observados y esperados.

### 4.1 Serie diaria de Mataró --> Barcelona

In [ ]:

# Obtener la movilidad diaria entre Mataró y Barcelona
flujo_mataro_bcn = con.execute(
    """
    SELECT
        day,
        SUM(viajes) AS viajes
    FROM read_parquet(?)
    WHERE municipio_origen = '08121'
      AND municipio_destino = '08019'
    GROUP BY day
    ORDER BY day
    """,
    [str(M2_PATH)]
).df()

# Representar la evolución temporal
plt.figure(figsize=(14, 5))

plt.plot(
    flujo_mataro_bcn["day"],
    flujo_mataro_bcn["viajes"],
    linewidth=0.9
)

plt.title("Evolución diaria de viajes: Mataró → Barcelona (2023–2025)")
plt.xlabel("Fecha")
plt.ylabel("Viajes diarios publicados")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# Comparar las fechas de la relación con las fechas disponibles en M2
fechas_m2 = set(movilidad_diaria["day"])
fechas_mataro = set(flujo_mataro_bcn["day"])

fechas_ausentes = sorted(fechas_m2 - fechas_mataro)

print("Días disponibles en M2:", len(fechas_m2))
print("Días de la relación Mataró → Barcelona:", len(fechas_mataro))
print("Días sin registro para esta relación:", len(fechas_ausentes))

if fechas_ausentes:
    print("\nFechas sin registro:")
    for fecha in fechas_ausentes:
        print(fecha.date())


### 4.2. Referencia elegida: mediana de días del mismo día semanal

Para una fecha objetivo, la referencia es la mediana de viajes observados en días comparables.

Se fija la regla antes de evaluar: mismo día semanal, ventana de 42 días, mínimo 3 controles, sin festivos oficiales de Cataluña ni fechas del Excel. Se analizan únicamente lunes a viernes y se excluyen **julio y agosto tanto de los objetivos como de los controles**, por decisión del proyecto. Esta exclusión de verano no equivale a un calendario completo de vacaciones escolares ni descarta puentes o festivos municipales.

En evaluación predictiva solo se usan controles **anteriores**; el propio día y el futuro quedan excluidos. En el Notebook 04 se distinguirá esa evaluación de la comparación retrospectiva con ±42 días.

La variación porcentual se calcula comparando los viajes observados el día de la incidencia con la mediana de viajes de los días comparables:

$$
\text{Variación (\%)} =
\left(
\frac{\text{Viajes observados}}{\text{Mediana de viajes comparables}}
- 1
\right)\times 100
$$

La referencia es descriptiva de la movilidad agregada de todos los modos de transporte. No representa un escenario contrafactual sin incidencia ni permite estimar el número de pasajeros ferroviarios afectados.

Los datos ausentes no se imputan. Si la mediana de los días comparables es cero, la variación porcentual se considera no disponible.

In [ ]:
festivos_cat = holidays.country_holidays("ES", subdiv="CT", years=[2023, 2024, 2025])
INCIDENCIAS_PATH = PROJECT_ROOT / "data/raw/rodalies/RMOVE_Incidencias_Barcelona_Jul2024_Ago2025_v07.xlsx"
incidencias = cargar_catalogo(INCIDENCIAS_PATH)
fechas_con_incidencias = set(incidencias["date"].dropna())

# Se conservan las siete OD originales y se incorporan las veinte del corredor sur.
relaciones = {
    "Mataró → Barcelona": ("08121", "08019"),
    "Granollers → Barcelona": ("08096", "08019"),
    "Castelldefels → Barcelona": ("08056", "08019"),
    "La Garriga → Barcelona": ("08088", "08019"),
    "Sabadell → Barcelona": ("08187", "08019"),
    "Terrassa → Barcelona": ("08279", "08019"),
    "Martorell → Granollers": ("08114", "08096"),
}
municipios_sur = {"08019": "Barcelona", "08169": "El Prat de Llobregat",
                  "08301": "Viladecans", "08089": "Gavà", "08056": "Castelldefels"}
for origen, no in municipios_sur.items():
    for destino, nd in municipios_sur.items():
        if origen != destino:
            relaciones[f"{no} → {nd}"] = (origen, destino)
movilidad_od = cargar_relaciones(con, M2_PATH, relaciones)
print("Relaciones:", movilidad_od["Relacion"].nunique())
print("Meses excluidos:", MESES_EXCLUIDOS)

### 4.3. Evaluación predictiva sin información del futuro

Para comprobar si la mediana representa adecuadamente la movilidad habitual, la comparamos con otros dos métodos: la media y el valor del último día comparable.

La evaluación se realiza sobre fechas de 2025 con datos válidos, de lunes a viernes, excluyendo julio, agosto, festivos y días con incidencias registradas. Para cada fecha y relación origen-destino, los tres métodos utilizan los mismos días comparables de los 42 días anteriores, sin incorporar datos de viajes futuros.

Comparamos las referencias calculadas con los viajes realmente observados mediante cuatro métricas: MAE (error absoluto medio), WAPE (error porcentual absoluto ponderado), sesgo (tendencia a sobreestimar o subestimar) y cobertura (proporción de observaciones con referencia válida).

Esta evaluación permite determinar cómo se comporta la mediana frente a otras alternativas y valorar su utilidad para detectar variaciones de movilidad. No obstante, la ausencia de incidencias registradas no garantiza que todos los días evaluados presenten condiciones habituales.

In [ ]:
validacion = evaluar_mediana(movilidad_od, fechas_con_incidencias, festivos_cat)
assert not validacion.empty
assert (validacion["ultima_fecha_control"] < validacion["day"]).all()
assert validacion["n_controles"].ge(3).all()
assert ~validacion["day"].dt.month.isin(MESES_EXCLUIDOS).any()
assert validacion["day"].dt.day_of_week.lt(5).all()

tabla_metodos = pd.DataFrame([
    {"Referencia": metodo, **metricas_referencia(validacion, metodo)}
    for metodo in ["mediana", "media", "ultimo_comparable"]
])
display(tabla_metodos.round(2))
tabla_territorial = pd.DataFrame([
    {"Relacion": nombre, **metricas_referencia(grupo, "mediana"),
     "n_candidatos": int(grupo["n_candidatos"].iloc[0])}
    for nombre, grupo in validacion.groupby("Relacion")
])
display(tabla_territorial.round(2))

En esta evaluación, la mediana obtiene los mejores resultados globales, con un WAPE del 7,98% y un MAE de 1.057 viajes, superando la media y al último dia comparable. Esto respalda la elección como método de referencia.

Por relaciones origen-destino, los errores varían considerablemente. La mayoría presentan un WAPE inferior al 12 %, aunque algunas relaciones de menor volumen, como Martorell → Granollers, alcanzan errores más elevados (25,44 %).

### 4.4. Mataró → Barcelona: viajes y referencia comparable

Este gráfico compara los viajes diarios observados entre Mataró y Barcelona con la movilidad habitual estimada mediante la mediana de los días comparables de las seis semanas anteriores.

También se representa la banda intercuartílica (P25–P75), que contiene el 50 % central de los valores de los días utilizados como referencia. Esta banda permite visualizar la variabilidad habitual de la movilidad e identificar fechas en las que los viajes observados se alejan de los valores más habituales.

Solo se incluyen fechas que cumplen los criterios de selección y disponen de suficientes días comparables. Las desviaciones observadas no implican necesariamente la existencia de una anomalía estadísticamente significativa.

In [ ]:
serie = validacion.loc[validacion["Relacion"].eq("Mataró → Barcelona")].sort_values("day")
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(serie["day"], serie["viajes_observados"], label="Observados", linewidth=1)
ax.plot(serie["day"], serie["mediana"], label="Mediana de días anteriores", linewidth=1)
ax.fill_between(serie["day"], serie["p25_viajes_controles"],
                serie["p75_viajes_controles"], alpha=.18, label="P25–P75 de viajes")
ax.set(title="Mataró → Barcelona · laborables fuera de julio/agosto",
       ylabel="Viajes", xlabel="Fecha")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

Los viajes observados entre Mataró y Barcelona siguen generalmente una evolución similar a la mediana de referencia, situándose habitualmente dentro o cerca de la banda intercuartílica (P25–P75).

No obstante, se identifican algunas desviaciones importantes, especialmente en abril y mayo, con caídas puntuales muy por debajo de los valores habituales. Estas diferencias podrían deberse a acontecimientos excepcionales, incidencias no registradas o problemas de cobertura de los datos.


In [ ]:
# Calcular la variación porcentual respecto a la mediana
desviaciones = serie.copy()

desviaciones["variacion_pct"] = (
    (desviaciones["viajes_observados"] / desviaciones["mediana"] - 1) * 100
)

# Seleccionar las 5 mayores caídas y los 5 mayores aumentos
mayores_caidas = desviaciones.nsmallest(5, "variacion_pct")
mayores_aumentos = desviaciones.nlargest(5, "variacion_pct")

# Unir y ordenar los resultados
tabla = pd.concat([mayores_caidas, mayores_aumentos])
tabla = tabla.sort_values("variacion_pct")

# Mostrar únicamente las columnas relevantes
tabla = tabla[[
    "day",
    "viajes_observados",
    "mediana",
    "variacion_pct"
]].copy()

tabla["day"] = pd.to_datetime(tabla["day"]).dt.strftime("%d/%m/%Y")

display(
    tabla.style
    .format({
        "viajes_observados": "{:,.0f}",
        "mediana": "{:,.0f}",
        "variacion_pct": "{:+.2f}%"
    })
    .background_gradient(subset=["variacion_pct"], cmap="RdYlGn")
)

## 5. Conclusiones

El análisis exploratorio confirma que la movilidad intermunicipal presenta patrones semanales y estacionales, por lo que es necesario utilizar referencias adaptadas al calendario.

La evaluación de los métodos de referencia muestra que la mediana de días comparables obtiene los menores errores globales (WAPE del 7,98 %), aunque su precisión varía según la relación origen-destino.

Por ello, se adopta la mediana como referencia para cuantificar las variaciones de movilidad en el Notebook 04, donde se analizarán incidencias ferroviarias concretas, considerando las limitaciones de los datos y sin atribuir causalidad directa a Rodalies.